# 02 · Train and submit

A LightGBM baseline, submitted to the **current round**. About ten minutes.

Coming from [`01_explore_the_data.ipynb`](01_explore_the_data.ipynb). The submission section
sends a real, scored submission only when you flip `SUBMIT = True`, so flip it deliberately.
Each model gets one submission per round: sending again while the window is open replaces it
in place, and once the round closes the entry is final.

In [ ]:
# Install only what is missing. Re-running this cell on a machine that already has
# the SDK should not be a network round-trip that tries to upgrade it.
try:
    import everestapi, lightgbm, pandas, pyarrow, numpy, scipy, matplotlib  # noqa: F401
    assert tuple(map(int, everestapi.__version__.split(".")[:3])) >= (0, 3, 38)
except (ImportError, AssertionError, ValueError):
    %pip install --quiet "everestapi>=0.3.40" lightgbm pandas pyarrow numpy scipy matplotlib

In [ ]:
import os
from everestapi import EverestAPI

# Read every credential from the environment, so nothing secret is written into the notebook.
base_url = os.environ.get("EIQ_BASE_URL", "https://api.everesteer.ai")
api_key = os.environ.get("EIQ_API_KEY") or os.environ.get("EVEREST_API_KEY")

# Fail fast with a clear message instead of a cryptic 401/403 deeper in the notebook.
if not api_key:
    raise RuntimeError("Set EIQ_API_KEY (onboarding -> Copy setup command).")

# Only the gated STAGING mirror needs a Cloudflare Access service token; the public
# site does not. The SDK picks the CF_ACCESS_* env vars up automatically when set.
if "staging" in base_url and not (
    os.environ.get("CF_ACCESS_CLIENT_ID") and os.environ.get("CF_ACCESS_CLIENT_SECRET")
):
    raise RuntimeError(
        "Staging is behind Cloudflare Access - set CF_ACCESS_CLIENT_ID / "
        "CF_ACCESS_CLIENT_SECRET, or point EIQ_BASE_URL at the public site."
    )

client = EverestAPI(api_key=api_key, base_url=base_url)
# Only `status` is meaningful here. The health probe is a constant-cost liveness
# check, so its `db` and `dataset` fields are always null by design.
print("connection:", client.health().get("status"))  # -> connection: ok

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# One small palette across every plot in these notebooks, so charts read as one system.
NAVY, TEAL, CORAL, GREY = "#09142F", "#007B63", "#EC9A5F", "#9AA3B2"
plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": GREY, "axes.grid": True, "grid.color": "#E6E9EF",
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 9, "axes.titlesize": 10, "axes.titleweight": "bold",
})

In [ ]:
import pandas as pd

schema = client.get_dataset_schema()
schema_verbose = client.get_dataset_schema(verbose=True)
feature_sets = schema_verbose["feature_sets"]
target_cols = list(schema["targets"])

# The graded column, from the schema - it differs between datasets and is NOT
# necessarily the first entry in `targets`.
PRIMARY_TARGET = schema["primary_target"]

# One value in the encoding means "missing" rather than a low bin. Read it; do not assume -1.
MISSING = (schema.get("feature_encoding") or {}).get("missing")
if MISSING is None:
    MISSING = -1.0

# download_dataset writes a parquet and returns its path. `train` is the labeled split.
train = pd.read_parquet(client.download_dataset(universe="futures", split="train"))

EXPED = "exped"   # each exped = one trading day

# Smallest published set: fewer features, faster. Some datasets publish small/medium/all
# tiers; this one publishes a single set, so min() returns it and there is no wider tier
# to move up to later.
FEATURE_SET_NAME = min(feature_sets, key=lambda n: len(feature_sets[n]))
feature_set = feature_sets[FEATURE_SET_NAME]

def exped_num(e):
    return int(str(e).split("_")[-1])   # "exped_8978" -> 8978, so expeds sort in time order

# Every 4th exped, purely for speed in this walkthrough. Use all of it for real models.
keep_expeds = sorted(train[EXPED].unique(), key=exped_num)[::4]
train = train[train[EXPED].isin(keep_expeds)]

print(f"train {train.shape}")
print(f"features used: {len(feature_set)} from the {FEATURE_SET_NAME!r} set "
      f"({len(feature_sets)} set(s) published)")

## 1. Hold out the tail of train

`train` is the only split you can score yourself: `validation` and `live` both ship with their
targets blank (validation is scored server-side on the practice board, live on the round). So
measure yourself on the end of `train`. Two rules:

- **Hold out the last expeds, not a random sample.** Random rows leak across time.
- **Embargo the boundary.** The target is a forward return that realises over 20 days, so the
  last fitting expeds already encode what happens over the first holdout expeds.
  `EMBARGO = 20` expeds covers that horizon (and more, since the setup cell keeps only every
  4th exped).

Training on Everesteer's servers has the same trap: a `train(...)` job fits the **whole** split
unless told otherwise. Pass `train_filter={"exped": {"cutoff_lt": <first embargo exped>}}` so the
fit stops before the embargo, as
[`futures_starter_hosted.py`](futures_starter_hosted.py) does.

In [ ]:
HOLDOUT_EXPEDS = 100   # the tail of train, kept aside
EMBARGO = 20           # expeds discarded between fit and holdout

ordered = sorted(train[EXPED].unique(), key=exped_num)
holdout_expeds = set(ordered[-HOLDOUT_EXPEDS:])
fit_expeds = set(ordered[: -(HOLDOUT_EXPEDS + EMBARGO)])

fit = train[train[EXPED].isin(fit_expeds)]
holdout = train[train[EXPED].isin(holdout_expeds)].copy()

print(f"fit     {len(fit):>8,} rows over {len(fit_expeds):>4} expeds")
print(f"embargo {'':>8}  {EMBARGO:>4} expeds discarded")
print(f"holdout {len(holdout):>8,} rows over {len(holdout_expeds):>4} expeds")
assert not (fit_expeds & holdout_expeds), "fit and holdout must not share expeds"

## 2. Train

Trees are the usual default here: they take binned integer features and missing values as they
are, and find interactions without scaling. Ridge is the cheap low-variance alternative and a
good ensemble partner; MLPs need more care on data this noisy.

Hosted `train` covers `lightgbm`, `xgboost`, `ridge`, `mlp` and `random_forest`, plus
`model="custom"` for your own code - see [`futures_starter_hosted.py`](futures_starter_hosted.py).

`features_matrix` maps the missing value to `NaN`, so LightGBM treats it as missing and not as
a real bin.

In [ ]:
import lightgbm as lgb

def features_matrix(df, cols, missing=MISSING):
    # Test the missing value itself rather than a range: a dataset that ever encodes a bin
    # below zero would break a range test silently. Bound as a default so it travels by
    # value with anything that pickles this.
    X = df[cols].astype("float32")
    return X.mask(X == missing)

# Lightly-regularized starting params, not tuned.
model = lgb.LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.01,
    max_depth=6,
    num_leaves=64,
    colsample_bytree=0.1,    # few features per tree -> more diverse trees, less overfit
    min_child_samples=500,   # large leaves -> smoother predictions, less overfit
    random_state=42,
    verbose=-1,
)
model.fit(features_matrix(fit, feature_set), fit[PRIMARY_TARGET])
print("Trained on", f"{len(fit):,}", "rows.")

## 3. Evaluate on the holdout

CORR is rank correlation between your predictions and the target, computed **within each exped**
and summarised across expeds. 

Sharpe is the sharpe ratio of CORR. Sharpe and max drawdown are display-only diagnostics: they say
whether the edge is consistent, but the board ranks on the round score, a weighted blend of CORR, AIMC and NCORR
(`explain_scoring` has the live weights).

In [ ]:
from scipy.stats import spearmanr

holdout["prediction"] = model.predict(features_matrix(holdout, feature_set))

def per_exped_corr(df):
    # Spearman (rank) correlation between prediction and target, one value per exped.
    out = {}
    for e, g in df.groupby(EXPED):
        rho, _ = spearmanr(g["prediction"], g[PRIMARY_TARGET])
        if np.isfinite(rho):
            out[e] = rho
    # Order by exped number so the cumulative curve and drawdown are chronological.
    s = pd.Series(out)
    return s.reindex(sorted(s.index, key=exped_num))

corr = per_exped_corr(holdout.dropna(subset=[PRIMARY_TARGET]))
cum = corr.cumsum()
peak = cum.expanding().max()
drawdown = peak - cum

print(f"Mean CORR {corr.mean():.4f} | Std {corr.std():.4f} | "
      f"Sharpe {corr.mean() / corr.std():.2f} | % positive {(corr > 0).mean() * 100:.0f}%")
print(f"Max drawdown {drawdown.max():.4f}")

In [ ]:
# Cumulative CORR with the drawdown from the running peak shaded underneath.
x = range(len(cum))
fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(9, 5), sharex=True, gridspec_kw={"height_ratios": [3, 1]}
)

ax1.plot(x, peak.values, color=GREY, lw=0.8, ls="--", label="running peak")
ax1.plot(x, cum.values, color=NAVY, lw=1.4, label="cumulative CORR")
ax1.fill_between(x, cum.values, peak.values, color=CORAL, alpha=0.25, label="drawdown")
ax1.axhline(0, color=GREY, lw=0.8)
ax1.set(title="Cumulative holdout CORR (embargoed tail of train)", ylabel="cumulative CORR")
ax1.legend(fontsize=7, frameon=False, loc="upper left")
ax1.margins(x=0)

ax2.fill_between(x, 0, -drawdown.values, color=CORAL, alpha=0.6)
ax2.set(xlabel="exped (chronological)", ylabel="drawdown", xticks=[])
ax2.margins(x=0)
plt.tight_layout(); plt.show()

## 4. How does it compare?

Both calls are read-only and safe to run at any time.

- `get_leaderboard()`, the tournament leaderboard. Read `rank_metric` on the response for
  what it is actually ordered by, and call `explain_scoring` for the current payout weights
  rather than assuming which term dominates; they are live platform settings and have changed.
- `download_benchmark("futures", "train")`, the benchmark model's predictions. **AIMC is
  measured against the benchmark model**, so it is the baseline to beat. Our holdout came out
  of `train`, so the comparison is like-for-like.

In [ ]:
try:
    lb = client.get_leaderboard()
    entries = lb.get("entries") or lb.get("leaderboard") or []
    print(f"Leaderboard: {len(entries)} entries | ranked on: {lb.get('rank_metric', 'unknown')}")
    for e in entries[:5]:
        name = e.get("model_name") or e.get("agent_name") or e.get("name") or "-"
        score = e.get("mean_payout")
        print(f"  #{e.get('rank', '?'):>3}  {name:<32}"
              + (f"  {score:.4f}" if isinstance(score, (int, float)) else ""))
except Exception as e:
    print("Leaderboard unavailable:", e)

try:
    bench = pd.read_parquet(client.download_benchmark("futures", "train"))
    baseline = pd.DataFrame({
        "benchmark": bench[[c for c in bench.columns if c != EXPED]].mean(axis=1),
        "bench_exped": bench[EXPED],
    })
    # Join on the id index, and keep a row only where the exped agrees too: ids repeat
    # across expeds, and a benchmark built from an older train file can share ids with
    # this one on different expeds, so an id-only join would score it against the wrong rows.
    rows = holdout.dropna(subset=[PRIMARY_TARGET]).join(baseline, how="inner")
    rows = rows[rows["bench_exped"] == rows[EXPED]]
    if len(rows) < 0.5 * len(holdout):
        raise ValueError(
            f"only {len(rows):,} of {len(holdout):,} holdout rows match on id and exped, so "
            "the benchmark was built from a different train file than the one served now"
        )
    bench_corr = pd.Series({
        e: spearmanr(g["benchmark"], g[PRIMARY_TARGET])[0] for e, g in rows.groupby(EXPED)
    }).replace([np.inf, -np.inf], np.nan).dropna()
    print(f"\nBenchmark mean CORR : {bench_corr.mean():+.4f}"
          f"  ({len(rows):,} of {len(holdout):,} holdout rows matched)")
    print(f"This baseline       : {corr.mean():+.4f}")
except Exception as e:
    print("\nBenchmark unavailable:", e)

## 5. Submit to the current round

Two ways to send predictions in, and they score different splits:

| lane | tool | what it scores |
|---|---|---|
| the tournament round. What you are ranked and paid on | `submit_futures_predictions` | the current round, on the `live` split |
| the practice board, open at any time | `submit_validation_diagnostics` | the fixed `validation` split, scored server-side |

**They take the same arguments, and they are disjoint `id` namespaces.** The practice board
always scores `validation`, so the `live` frame built below matches none of its ids: send it
there and the upload is accepted, then fails on zero id overlap. Rehearsing is worth it, but
rehearse with predictions made *on `validation`*; predict `live` only for the round itself.

- **Submit while the window is open.** Between rounds an upload is accepted but not scored.
- **Predict `live`, freshly downloaded.** It is one round, so predict every row (at most 200);
  the ids are the parquet index. Each round serves fresh ids, so a frame built for an earlier
  round matches nothing.
- **Validate before you submit.** `validate_submission(...)` runs the same coverage/range/
  finiteness checks as the real path, for free, without writing anything.
- **Rank-normalise into `[0, 1]`.** Futures predictions must be finite and within `[0, 1]`:
  the SDK raises locally and the platform returns a 400 otherwise. A raw regressor's output is
  not in that range. Only the ordering is scored, so `.rank(pct=True)` costs nothing; clipping
  would flatten the ordering instead.
- **One submission per model per round, replaced in place.** Sending again while the window is
  open replaces it (`replaced: true`, same `submission_id`); after the close a resend is a 409
  and the entry is final.
- **Several models?** The Python client has no batch method, so loop. Over MCP,
  `submit_futures_predictions_batch` takes 1 to 25 models per call; each item succeeds or fails
  on its own, so read `all_succeeded` and join the items back by `index`.
- **The model file is optional.** A daily submission is just the predictions dict. Uploading a
  cloudpickled `predict()` with `upload_model` only matters if you want the platform to predict
  for you (see [`futures_starter.py`](futures_starter.py)); `get_started` reports whether that hosted run is active.

In [ ]:
# Model names are public on the leaderboard, so pick one that does not describe your recipe.
# create_model(name=...) is idempotent WITH a name: re-running returns the existing model
# (status="already_exists"). Omit the name and every call creates a NEW model.
MODEL_NAME = "notebook-baseline"
SUBMIT = False    # flip to True when you're ready to spend the round's scored submission

# A model must EXIST before you can submit for it. Keep the response's `id`: that is the
# stable model_id. Reuse the same model every round so its history stays on one entry.
MODEL_ID = client.create_model(name=MODEL_NAME)["id"] if SUBMIT else None
if MODEL_ID:
    print(f"Model {MODEL_NAME!r} -> {MODEL_ID}")

# Which round are these rows for? A 404 means no round is active at all.
try:
    current = client.get_current_round(tournament="futures") or {}
except Exception as e:
    if getattr(e, "status_code", None) != 404:
        raise
    current = {}
ROUND = current.get("exped")   # pass it explicitly; never the literal "current"
print(f"round {ROUND!r} | window {current.get('submission_window')!r}")

if current.get("submission_window") != "open":
    # Between rounds (or with no active round) there is nothing to score: an upload now
    # is accepted but NOT scored. Rehearse on the practice board instead, and come back
    # when the next round opens.
    print("No round window is open right now: skipping the live prediction and submit.")
else:
    # The current round serves its ids on the LIVE split, as the parquet index. Re-download it
    # every round.
    live = pd.read_parquet(client.download_dataset(universe="futures", split="live"))
    if ROUND is not None and set(live[EXPED].unique()) != {ROUND}:
        raise RuntimeError(f"live carries {sorted(live[EXPED].unique())}, not round {ROUND!r}; re-download")
    raw = pd.Series(model.predict(features_matrix(live, feature_set)), index=live.index)

    # Both calls below take an {id: prediction} dict. Rank-normalise into [0, 1]: only the
    # ordering is scored, so this costs nothing, and clipping would flatten the ordering.
    preds = raw.rank(pct=True).to_dict()
    assert len(preds) == len(live), "one prediction per live row, no duplicate ids"
    print(f"Predictions: {len(preds):,} ids, first {next(iter(preds))!r}")

    # Free dry run: the same coverage/range/finiteness checks as the real submit, writes nothing.
    check = client.validate_submission(preds, exped=ROUND)
    print("validate_submission:", check)

    if SUBMIT:
        if check.get("issues"):
            print("Fix the issues validate_submission reported before submitting.")
        else:
            # No target= argument: the round is scored on the schema's primary target.
            result = client.submit_futures_predictions(MODEL_ID, preds, exped=ROUND)
            print("Submitted:", result)
            print("submission status:", client.get_submission_status(
                tournament="futures", round=ROUND, model_id=MODEL_ID))

## 6. Read your result, then go again

- `get_submission_status(tournament="futures", round=<exped>, model_id=...)` confirms your
  submission landed; it only ever returns your own submissions.
- `get_leaderboard()`, the tournament leaderboard.
- `get_scores(model_id)` once the round resolves; the target realises over 20 days.

Then go again the next weekday: try different feature sets, targets, and models, and read
scores as each round resolves.

In [ ]:
try:
    lb = client.get_leaderboard()
    rows = lb.get("entries") or lb.get("leaderboard") or []
    print(f"Leaderboard: {len(rows)} entries")
    for r in rows[:10]:
        name = r.get("model_name") or r.get("agent_name") or r.get("name") or "-"
        score = r.get("mean_payout")
        print(f"  #{r.get('rank', '?'):>3}  {name:<32}"
              + (f"  {score:.4f}" if isinstance(score, (int, float)) else ""))
except Exception as e:
    print("Leaderboard unavailable:", e)

if MODEL_ID:
    print("\nsubmission status:", client.get_submission_status(
        tournament="futures", round=ROUND, model_id=MODEL_ID))

## What to try next

- **Use all of the data.** The setup cell keeps every 4th exped for speed. This dataset
  publishes a single feature set (`all`), so there is no wider tier to move to: the lever here
  is expeds, not feature sets.
- **Auxiliary targets.** Train on the dataset's other targets and ensemble; drop NaN-target
  rows, and keep only one of any near-inverse pair.
- **Feature neutralization.** Reducing exposure to the features your model leans on most can
  lift NCORR and AIMC.
  NCORR neutralizes against 10 core features (`core_feature_overlap` in the schema), and all 10
  sit inside `all`, so you cannot avoid them by picking features. The
  [`eiq-model-implementation`](../.claude/skills/eiq-model-implementation/SKILL.md) skill carries
  the projection helper and the offline AIMC proxy that scores a neutralization sweep.
- **Fit on Everesteer's servers.** [`futures_starter_hosted.py`](futures_starter_hosted.py) trains
  the same baseline on Everesteer compute with server-side exped-purged CV. A model file is
  optional either way: a daily submission is just the predictions.
- **Optimise the round score, not CORR alone.** `explain_scoring` reports the live weights.
- **Stake, deliberately.** Staking is optional and real money: stakes lock at the end of the
  daily round and returns arrive after 20 days. See the
  [`eiq-futures-submission`](../.claude/skills/eiq-futures-submission/SKILL.md) skill.

Full agent contract and research skills: [`AGENTS.md`](../AGENTS.md).